<h2>2026 NZF Impacts on Caribbean - 'mapping_ex_v0.1.ipynb'</h2>

The initial EU ETS script looked at what EU ETS costs were likely to be for African states. This notebook goes one further and introduces NZF impact costs, using Scenario 24 of DNV's CIA Task 2 work, specifically reverse engineering policy impact onto the 4th IMO GHG Study dataset using DNV-derived Cost projections per unit of maritime transport supply (US$/dwt.nm). 

Forked from '/Users/apple/repos/PubEnv/2026_eu_ets_africa/2026_eu_ets_africa_v0.1.ipynb' on 24th March 2026, this 'mapping' script can be used to match 'exporter-side' trade flows to the maritime shipping network.

In [1]:
# Import packages
import os
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None

<u><h2>Reading in the 'Activity' Dataset...</h2></u>

In [2]:
voys_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}
voys = pd.read_csv("/Users/apple/repos/CaribbEnv/2026/activity/activity_v0.1.csv", dtype=voys_d)
voys.head(2)

,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,dwtnm,bau_usd_23,bau_usd_30,bau_usd_40,bau_usd_50,s24_usd_23,s24_usd_30,s24_usd_40,s24_usd_50,ets_coverage,ets_23_usd,ets_30_usd
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,5.806091e+07,159128.795463,165190.366278,133758.799530,132626.163678,159128.795463,191097.374915,216464.893320,228787.351725,0.0,0.0,0.0
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,7.401583e+06,20285.679803,21058.406602,17051.522135,16907.134137,20285.679803,24361.022450,27594.864284,29165.726710,0.0,0.0,0.0


<u><h2>Reading in the 'Trade' Dataset ...</h2></u>

In [3]:
trade_ex_d = {
    "trade_idx":"int", 
    "source_iso_code":"str", "source_country":"str", "dest_iso_code":"str", "dest_country":"str",	
    "HS2":"str", "mode_code":"str", "mode_label":"str", "val_usd":"float", "vol_kg":"float"
}
trade_ex = pd.read_csv("/Users/apple/repos/CaribbEnv/2026/trade/trade_ex_v0.1.csv", dtype=trade_ex_d)
trade_ex.head(2)

,trade_idx,source_iso_code,source_country,dest_iso_code,dest_country,HS2,mode_code,mode_label,val_usd,vol_kg
0,0,004,Afghanistan,012,Algeria,08,21,Sea,0.000,0.000
1,1,004,Afghanistan,012,Algeria,13,21,Sea,101.048,72.711


# 1. Read-in Additional Datasets

These include:

<ul>
    <li><b>Country ISO Codes</b>: A table composed of country names and their ISO references is read in for use in the matching process.</li>
    <li><b>Facilitating Country Mapping</b>: Given that many countries may connect to the international shipping transport network via the facilities of neighbouring countries, a second table is read-in that provides the ISO codes of countries that may be facilitate the target countries international maritime trade.</li>
    <li><b>Commodity-Vessel Type Mapping</b>: Finally, only certain vessel types are physically able to facilitate the transportation of certain goods. Therefore, this table is read-in that maps commodities (at a Harmonised System Level 2 resolution) to specific vessel types observed in the IMO IV dataset, based on Bills of Lading data curated as part of the Global Shipping Watch project.</li>
</ul>

## 1.1 Country ISO Codes

A table composed of country names and their ISO references is read in for use in the matching process.

In [7]:
# Read-in ISO Country Labels dataset and use as Standard !! Update for National stats, or whatever underpins 'voys_2018_foc.csv'?
iso_codes_c = ["name", "alpha-2", "alpha-3", "country-code"]
iso_codes_d = {"alpha-2":str, "alpha-3":str, "country-code":str}
iso_codes_r = {"name":"iso_country", "alpha-2":"iso_2", "alpha-3":"iso_3", "country-code":"iso_code"}
iso_codes = pd.read_csv("/Users/apple/repos/datasets/national_stats/country_iso_codes.csv", usecols=iso_codes_c, dtype=iso_codes_d).rename(columns=iso_codes_r)
iso_codes.loc[iso_codes.iso_country == "Namibia", "iso_2"] = "NA"
display(iso_codes.iloc[:2])

,iso_country,iso_2,iso_3,iso_code
0,Afghanistan,AF,AFG,004
1,Åland Islands,AX,ALA,248


## 1.2 Facilitating Country Mapping

Given that many countries may connect to the international shipping transport network via the facilities of neighbouring countries, a second table is read-in that provides the ISO codes of countries that may be facilitate the target countries international maritime trade.

In [5]:
# Read-in mapping of trade partner countries to facilitating country lists
fac_map_d = {"iso_2":"str", "iso_3":"str", "iso_code":"str", "facilitating_countries":"str"}
fac_map = pd.read_csv("/Users/apple/repos/AISenv/bilat_to_shipping_mapping/bilat_to_shipping_mapping_v0.2.csv", dtype=fac_map_d)
display(fac_map.iloc[:2])

,iso_country,iso_2,iso_3,iso_code,lldc,sids,facilitating_countries,fac_country_names
0,Afghanistan,AF,AFG,004,1,0,364 586,"Iran (Islamic Republic of), Pakistan"
1,Åland Islands,AX,ALA,248,0,0,248,Åland Islands


## 1.3 Commodity to Vessel Type Mapping

Finally, only certain vessel types are physically able to facilitate the transportation of certain goods. Therefore, this table is read-in that maps commodities (at a Harmonised System Level 2 resolution) to specific vessel types observed in the IMO IV dataset, based on Bills of Lading data curated as part of the Global Shipping Watch project.

In [6]:
# Merge with dataset of most common facilitating vessel types
hs_to_vess_type_d = {"hs2":"float", "description":"str", "ship_type_standardized":"str"}
hs_to_vess_type = pd.read_csv("/Users/apple/repos/AISenv/trade_voyage_linking/gsw_vessel_type_mapping_v0.1.csv", dtype=hs_to_vess_type_d)
display(hs_to_vess_type.iloc[:2])

,hs2,description,ship_type_standardized,fob_us_pct,fob_us,kg_pct,kg
0,1.0,Animals; live,General cargo,90.336503,7.158340e+08,84.014207,2.256197e+08
1,1.0,Animals; live,Container,9.470854,7.504784e+07,15.340383,4.119651e+07


# 2. Matching between Trade and Voyages

## 2.1 Define Countries to Run in Terms of ISO Code

In [9]:
def return_list_without_names_in_dir(directory, list_x):
    to_remove = [i[:3] for i in os.listdir(directory) if i not in [".DS_Store"]]
    for i in to_remove:
        try:
            list_x.remove(i)
        except:
            pass
    return pd.Series(list_x).sort_values().values
    
# Set up for Exporter-side Run
exp_side_dir = "/Users/apple/repos/CaribbEnv/2026/trade_activity_mapping/mapping_ex/"

trade_countries = list(set(list(trade_ex.source_iso_code.values) + list(trade_ex.dest_iso_code.values)))
print("Number of unique countries present in the Trade Dataset: {0}.".format(len(trade_countries)))

exp_isos_to_run = return_list_without_names_in_dir(exp_side_dir, trade_countries)
print("Number of unique countries present in the ISO codes to be run: {0}.".format(len(exp_isos_to_run)))
print("\nCountry ISO codes to be run:\n\n", exp_isos_to_run)

Number of unique countries present in the Trade Dataset: 222.
Number of unique countries present in the ISO codes to be run: 222.

Country ISO codes to be run:

 ['004' '008' '012' '016' '020' '024' '028' '031' '032' '036' '040' '044'
 '048' '050' '051' '052' '056' '060' '064' '068' '070' '072' '076' '084'
 '090' '092' '096' '100' '104' '108' '112' '116' '120' '124' '132' '136'
 '140' '144' '148' '152' '156' '158' '162' '170' '174' '180' '184' '188'
 '191' '192' '196' '203' '204' '208' '212' '214' '218' '222' '226' '231'
 '232' '233' '234' '238' '242' '246' '250' '258' '262' '266' '268' '270'
 '275' '276' '288' '292' '296' '300' '304' '308' '316' '320' '324' '328'
 '332' '340' '344' '348' '352' '356' '360' '364' '368' '372' '376' '380'
 '384' '388' '392' '398' '400' '404' '408' '410' '414' '417' '418' '422'
 '426' '428' '430' '434' '440' '442' '446' '450' '454' '458' '462' '466'
 '470' '478' '480' '484' '496' '498' '499' '500' '504' '508' '512' '516'
 '520' '524' '528' '531' '533' '534

## 2.2 Run Matching process

In [10]:
counter, voys_x_trades_cols = 0, ["voy_idx", "trade_idx", "voy_x_trade_kg", "voy_x_trade_usd"]

for exp_iso in exp_isos_to_run:

    # Create output folder
    iso_name = iso_codes[(iso_codes.iso_code == exp_iso)].iso_country.values[0]
    iso_id = exp_iso + " - " + iso_name
    print("Processing Export Trade Flows for {0} (country {1} of {2}).".format(iso_id, counter + 1, len(exp_isos_to_run))) if counter in range(0, 300, 20) else False
    counter += 1

    # Check we have what we need for this exporter country
    if fac_map[fac_map.iso_code == exp_iso].shape[0] == 0:
        print("No mapping to shipping supply countries available for '{0}'.".format(iso_id))
        continue

    # Create directory if necessary
    if not os.path.exists(exp_side_dir+"{0}/".format(iso_id)):
        os.makedirs(exp_side_dir+"{0}/".format(iso_id)) 
    
    # Take a subset of the Trade dataset for where Trades records originate in the Origin Country
    trade_ex_iso = trade_ex[(trade_ex.source_iso_code == exp_iso)]

    # Isolate ISO codes of Source Countries assumed to accomodate the trade on behalf of the Exporter
    exp_fac_isos = fac_map[fac_map.iso_code == exp_iso].facilitating_countries.values[0].split()
    
    # Subset the Voyages dataset for those originating in these countries
    voys_exp = voys.loc[(voys.source_iso_code.isin(exp_fac_isos))]

    
    for HS2 in pd.Series(trade_ex_iso.HS2.unique()).sort_values().values:

        voys_x_trades_exp_HS2_ls = []
        trade_ex_iso_HS2 = trade_ex_iso[(trade_ex_iso.HS2 == HS2)]

        ## Take forward vessel types identified as facilitating at least 25% of the HS2 commodity in GSW data, then subset the Voyages TCM dataset for these vessel types only
        supply_vess_types = hs_to_vess_type[(hs_to_vess_type.hs2 == float(HS2)) & (hs_to_vess_type.fob_us_pct > 25.0)].ship_type_standardized.unique()
        voys_exp_vess_type = voys_exp.loc[(voys_exp["Vessel Type"].isin(supply_vess_types))]
        
        for trade_idx, trade in trade_ex_iso_HS2.iterrows():

            # Try to take ISO codes of countries assumed to accomodate the trade on behalf of the Importer.
            imp_fac_isos = fac_map[fac_map.iso_code == trade.dest_iso_code]
            if imp_fac_isos.shape[0] == 0:
                pass # No mapping to a facilitating import country available for this bilateral trade flow
            
            else:
                # Then subset the Voyages dataset for voyages arriving into these accomodating countries.
                voys_exp_vess_type_imp = voys_exp_vess_type.loc[(voys_exp_vess_type.dest_iso_code.isin(imp_fac_isos.facilitating_countries.values[0].split()))]
                if voys_exp_vess_type_imp.shape[0] == 0: 
                    pass # No Voyage data for this specific bilateral trade.
                
                else:
                    # Assume that the weight and value of the country- and HS2-specified trade flow is split evenly across each identified voyage
                    kg_on_voy = trade.vol_kg / voys_exp_vess_type_imp.shape[0]
                    usd_on_voy = trade.val_usd / voys_exp_vess_type_imp.shape[0]
        
                    # Catalogue the equal proportion of the trade weight and volume on supply voyages using their Trade and TCM indexes.
                    for sup_voy_idx, sup_voy in voys_exp_vess_type_imp.iterrows():        
                        voys_x_trades_exp_HS2_ls.append(pd.DataFrame([[sup_voy.voy_idx, trade.trade_idx, kg_on_voy, usd_on_voy]], columns=voys_x_trades_cols))

        if len(voys_x_trades_exp_HS2_ls) > 0:
            pd.concat(voys_x_trades_exp_HS2_ls, axis=0).to_csv(exp_side_dir+"{0}/{1}.csv".format(iso_id, str(HS2)), index=False)

Processing Export Trade Flows for 004 - Afghanistan (country 1 of 222).
Processing Export Trade Flows for 070 - Bosnia and Herzegovina (country 21 of 222).
Processing Export Trade Flows for 156 - China (country 41 of 222).
No mapping to shipping supply countries available for '158 - Taiwan, Province of China'.
Processing Export Trade Flows for 232 - Eritrea (country 61 of 222).
Processing Export Trade Flows for 316 - Guam (country 81 of 222).
Processing Export Trade Flows for 400 - Jordan (country 101 of 222).
Processing Export Trade Flows for 470 - Malta (country 121 of 222).
Processing Export Trade Flows for 548 - Vanuatu (country 141 of 222).
Processing Export Trade Flows for 624 - Guinea-Bissau (country 161 of 222).
Processing Export Trade Flows for 702 - Singapore (country 181 of 222).
Processing Export Trade Flows for 776 - Tonga (country 201 of 222).
Processing Export Trade Flows for 887 - Yemen (country 221 of 222).


## 2.3 Compile All National Datasets into One

In [11]:
counter, iso_codes, voys_x_trades_exp = 0, os.listdir(exp_side_dir), []

for iso_code in iso_codes:
    if iso_code == ".DS_Store":
        continue

    print("Reading in {0} (Country {1} of {2}).".format(iso_code, counter, len(iso_codes))) if counter in range(0, 300, 20) else False
    counter += 1
    HS2s = os.listdir(exp_side_dir+"{0}/".format(iso_code))
    
    voys_x_trades_HS2s_ls = []
    for HS2 in HS2s:
        voys_x_trades_HS2s_ls.append(pd.read_csv(exp_side_dir + "{0}/{1}".format(iso_code, HS2)))
    
    if len(voys_x_trades_HS2s_ls) > 0:
        voys_x_trades_exp.append(pd.concat(voys_x_trades_HS2s_ls, axis=0))

voys_x_trades_exp_all = pd.concat(voys_x_trades_exp, axis=0)
voys_x_trades_exp_all.to_csv(exp_side_dir + "voys_x_trades_exp_all.csv", index=False)
print("Weight of trade once linked with voyages: {0} of {1} million tonnes in original 'trade_ex' dataset ({2}%).".format(\
    round(voys_x_trades_exp_all.voy_x_trade_kg.sum() / 1e9, 1), \
    round(trade_ex.vol_kg.sum() / 1e9, 1), \
    round(100 * voys_x_trades_exp_all.voy_x_trade_kg.sum() / trade_ex.vol_kg.sum(), 1)
))
print("Value of trade once linked with voyages: ${0}bn of ${1}bn in original 'trade_ex' dataset ({2}%).".format(\
    round(voys_x_trades_exp_all.voy_x_trade_usd.sum() / 1e9, 1), \
    round(trade_ex.val_usd.sum() / 1e9, 1), \
    round(100 * voys_x_trades_exp_all.voy_x_trade_usd.sum() / trade_ex.val_usd.sum(), 1)
))
display(voys_x_trades_exp_all.head(2))

Reading in 140 - Central African Republic (Country 0 of 222).
Reading in 862 - Venezuela (Bolivarian Republic of) (Country 20 of 222).
Reading in 184 - Cook Islands (Country 40 of 222).
Reading in 580 - Northern Mariana Islands (Country 60 of 222).
Reading in 462 - Maldives (Country 80 of 222).
Reading in 266 - Gabon (Country 100 of 222).
Reading in 344 - Hong Kong (Country 120 of 222).
Reading in 478 - Mauritania (Country 140 of 222).
Reading in 524 - Nepal (Country 160 of 222).
Reading in 418 - Lao People's Democratic Republic (Country 180 of 222).
Reading in 654 - Saint Helena, Ascension and Tristan da Cunha (Country 200 of 222).
Reading in 352 - Iceland (Country 220 of 222).
Weight of trade once linked with voyages: 5687.9 of 6331.6 million tonnes in original 'trade_ex' dataset (89.8%).
Value of trade once linked with voyages: $5519.2bn of $6537.5bn in original 'trade_ex' dataset (84.4%).


,voy_idx,trade_idx,voy_x_trade_kg,voy_x_trade_usd
0,375358,89331,79.2828,10215.5396
1,375360,89331,79.2828,10215.5396


## 2.4 Reconstruct in Terms of the Original Dataset

In [13]:
# Introduce the vessel Dadweight Tonnage, CO2 and ETS Compliance costs associated with each voyage
voys_tm_cols = [
    "voy_idx", "deadweight", 
    "energy_tj", "co2_t", "ets_23_usd", "ets_30_usd",
    "bau_usd_23", "bau_usd_30", "bau_usd_40", "bau_usd_50", 
    "s24_usd_23", "s24_usd_30", "s24_usd_40", "s24_usd_50"
]
voys_x_trades_exp_all_voys_info_m = pd.merge(\
    voys_x_trades_exp_all, voys[voys_tm_cols], \
    left_on="voy_idx", right_on="voy_idx", \
    how="inner", indicator="_exp_voys_info_m"
)
print("\nPerformance of merge between 'voys_x_trades_exp_all' and 'voys': \n\n{0}.\n".format(\
    voys_x_trades_exp_all_voys_info_m._exp_voys_info_m.value_counts()))

# Evaluate proportion of ETS Compliance Costs for each 'voy_x_trade' by comparing its weight with the total weight estimated to be on that voyage across all other trades.
voys_x_trades_exp_all_voys_info_m["voy_x_trade_ene_tj"] = voys_x_trades_exp_all_voys_info_m.energy_tj * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_co2_t"] = voys_x_trades_exp_all_voys_info_m.co2_t * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)

# EU ETS Costs
voys_x_trades_exp_all_voys_info_m["voy_x_trade_ets_23_usd"] = voys_x_trades_exp_all_voys_info_m.ets_23_usd * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_ets_30_usd"] = voys_x_trades_exp_all_voys_info_m.ets_30_usd * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)

# NZF BAU Costs
voys_x_trades_exp_all_voys_info_m["voy_x_trade_bau_usd_23"] = voys_x_trades_exp_all_voys_info_m.bau_usd_23 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_bau_usd_30"] = voys_x_trades_exp_all_voys_info_m.bau_usd_30 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_bau_usd_40"] = voys_x_trades_exp_all_voys_info_m.bau_usd_40 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_bau_usd_50"] = voys_x_trades_exp_all_voys_info_m.bau_usd_50 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)

# NZF S.24 (GFI Flex Only) Costs
voys_x_trades_exp_all_voys_info_m["voy_x_trade_s24_usd_23"] = voys_x_trades_exp_all_voys_info_m.s24_usd_23 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_s24_usd_30"] = voys_x_trades_exp_all_voys_info_m.s24_usd_30 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_s24_usd_40"] = voys_x_trades_exp_all_voys_info_m.s24_usd_40 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
voys_x_trades_exp_all_voys_info_m["voy_x_trade_s24_usd_50"] = voys_x_trades_exp_all_voys_info_m.s24_usd_50 * (0.001 * voys_x_trades_exp_all_voys_info_m.voy_x_trade_kg / voys_x_trades_exp_all_voys_info_m.deadweight)
display(voys_x_trades_exp_all_voys_info_m.iloc[:2])

# Group by trade and sum all 'voy_x_trade' components
voys_x_trades_exp_all_voys_info_m_by_trade_d = {
    "voy_x_trade_kg":"sum", "voy_x_trade_usd":"sum", 
    "voy_x_trade_ene_tj":"sum", "voy_x_trade_co2_t":"sum", "voy_x_trade_ets_23_usd":"sum", "voy_x_trade_ets_30_usd":"sum", 
    "voy_x_trade_bau_usd_23":"sum", "voy_x_trade_bau_usd_30":"sum", "voy_x_trade_bau_usd_40":"sum", "voy_x_trade_bau_usd_50":"sum",
    "voy_x_trade_s24_usd_23":"sum", "voy_x_trade_s24_usd_30":"sum", "voy_x_trade_s24_usd_40":"sum", "voy_x_trade_s24_usd_50":"sum"
}

# Rename with '_recon' suffix to reflect reconstruction of the database
voys_x_trades_exp_all_voys_info_m_by_trade_r = {
    "voy_x_trade_kg":"vol_kg_recon", "voy_x_trade_usd":"val_usd_recon", 
    "voy_x_trade_ene_tj":"ene_tj_recon", "voy_x_trade_co2_t":"co2_t_recon", "voy_x_trade_ets_23_usd":"ets_23_usd_recon", "voy_x_trade_ets_30_usd":"ets_30_usd_recon", 
    "voy_x_trade_bau_usd_23":"bau_usd_23_recon", "voy_x_trade_bau_usd_30":"bau_usd_30_recon", "voy_x_trade_bau_usd_40":"bau_usd_40_recon", "voy_x_trade_bau_usd_50":"bau_usd_50_recon",
    "voy_x_trade_s24_usd_23":"s24_usd_23_recon", "voy_x_trade_s24_usd_30":"s24_usd_30_recon", "voy_x_trade_s24_usd_40":"s24_usd_40_recon", "voy_x_trade_s24_usd_50":"s24_usd_50_recon"
}

voys_x_trades_exp_all_voys_info_m_by_trade = voys_x_trades_exp_all_voys_info_m.groupby(["trade_idx"]).agg(\
    voys_x_trades_exp_all_voys_info_m_by_trade_d).reset_index().rename(columns=voys_x_trades_exp_all_voys_info_m_by_trade_r)
display(voys_x_trades_exp_all_voys_info_m_by_trade.iloc[:2])


Performance of merge between 'voys_x_trades_exp_all' and 'voys': 

_exp_voys_info_m
both          64421696
left_only            0
right_only           0
Name: count, dtype: int64.



,voy_idx,trade_idx,voy_x_trade_kg,voy_x_trade_usd,deadweight,energy_tj,co2_t,ets_23_usd,ets_30_usd,bau_usd_23,bau_usd_30,bau_usd_40,bau_usd_50,s24_usd_23,s24_usd_30,s24_usd_40,s24_usd_50,_exp_voys_info_m,voy_x_trade_ene_tj,voy_x_trade_co2_t,voy_x_trade_ets_23_usd,voy_x_trade_ets_30_usd,voy_x_trade_bau_usd_23,voy_x_trade_bau_usd_30,voy_x_trade_bau_usd_40,voy_x_trade_bau_usd_50,voy_x_trade_s24_usd_23,voy_x_trade_s24_usd_30,voy_x_trade_s24_usd_40,voy_x_trade_s24_usd_50
0,375358,89331,79.2828,10215.5396,18322.0,1.194266,89.887274,0.0,0.0,39006.141379,35950.292490,34095.492095,31837.770800,39006.141379,41972.830301,59312.930427,62608.644677,both,0.000005,0.000389,0.0,0.0,0.168787,0.155564,0.147538,0.137768,0.168787,0.181624,0.256658,0.270920
1,375360,89331,79.2828,10215.5396,18322.0,0.920154,69.133785,0.0,0.0,41969.650624,38681.632233,36685.912539,34256.659846,41969.650624,45161.735080,63819.257160,67365.364792,both,0.000004,0.000299,0.0,0.0,0.181611,0.167383,0.158747,0.148235,0.181611,0.195423,0.276158,0.291503


,trade_idx,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,12,1.841,72.646,5.429581e-07,0.000042,0.0,0.000000,0.035696,0.035722,0.030116,0.028068,0.035696,0.039151,0.045789,0.047223
1,25,183.004,114.923,5.246432e-05,0.004064,0.0,0.487683,2.205752,2.289774,1.854088,1.838388,2.205752,2.648882,3.000512,3.171319


## 2.5 Evaluate Performance of Reconstructed Dataset

In [14]:
trade_ex_voys_info = pd.merge(\
    trade_ex, voys_x_trades_exp_all_voys_info_m_by_trade, \
    left_on="trade_idx", right_on="trade_idx", how="inner", \
    indicator="_recon_voys_info_merge")

print("\nPerformance of merge between 'trade_ex' and 'voys_x_trades_exp_all_voys_info_m_by_trade': \n\n{0}.\n".format(trade_ex_voys_info._recon_voys_info_merge.value_counts()))

print("Weight captured in reconstructed trade database: {0} of {1} million tonnes in original 'trade_ex' dataset ({2}%).".format(\
    round(trade_ex_voys_info.vol_kg_recon.sum() / 1e9, 1), round(trade_ex.vol_kg.sum() / 1e9, 1), \
    round(100 * trade_ex_voys_info.vol_kg_recon.sum() / trade_ex.vol_kg.sum(), 1)))

print("Value captured in reconstructed trade database: ${0}bn of ${1}bn in original 'trade_ex' dataset ({2}%).".format(\
    round(trade_ex_voys_info.val_usd_recon.sum() / 1e9, 1), round(trade_ex.val_usd.sum() / 1e9, 1),\
    round(100 * trade_ex_voys_info.val_usd_recon.sum() / trade_ex.val_usd.sum(), 1)))

print("Energy Demand captured in reconstructed trade database: {0} of {1} EJs in original 'voys' dataset ({2}%).".format(\
    round(trade_ex_voys_info.ene_tj_recon.sum() / 1e6, 1), round(voys.energy_tj.sum() / 1e6, 1), \
    round(100 * trade_ex_voys_info.ene_tj_recon.sum() / voys.energy_tj.sum(), 1)))

print("CO2 emissions captured in reconstructed trade database: {0} of {1} million tonnes in original 'voys' dataset ({2}%).".format(\
    round(trade_ex_voys_info.co2_t_recon.sum() / 1e6, 1), round(voys.co2_t.sum() / 1e6, 1), \
    round(100 * trade_ex_voys_info.co2_t_recon.sum() / voys.co2_t.sum(), 1)))

print("2030 ETS Compliance Costs captured in reconstructed trade database: ${0}bn of ${1}bn in original 'voys' dataset ({2}%).".format(\
    round(trade_ex_voys_info.ets_30_usd_recon.sum() / 1e9, 1), round(voys.ets_30_usd.sum() / 1e9, 1), \
    round(100 * trade_ex_voys_info.ets_30_usd_recon.sum() / voys.ets_30_usd.sum(), 1)))

print("2030 NZF BAU Compliance Costs captured in reconstructed trade database: ${0}bn of ${1}bn in original 'voys' dataset ({2}%).".format(\
    round(trade_ex_voys_info.bau_usd_30_recon.sum() / 1e9, 1), round(voys.bau_usd_30.sum() / 1e9, 1), \
    round(100 * trade_ex_voys_info.bau_usd_30_recon.sum() / voys.bau_usd_30.sum(), 1)))

print("2030 NZF S24 Compliance Costs captured in reconstructed trade database: ${0}bn of ${1}bn in original 'voys' dataset ({2}%).".format(\
    round(trade_ex_voys_info.s24_usd_30_recon.sum() / 1e9, 1), round(voys.s24_usd_30.sum() / 1e9, 1), \
    round(100 * trade_ex_voys_info.s24_usd_30_recon.sum() / voys.s24_usd_30.sum(), 1)))

display(trade_ex_voys_info.head(2))


Performance of merge between 'trade_ex' and 'voys_x_trades_exp_all_voys_info_m_by_trade': 

_recon_voys_info_merge
both          239138
left_only          0
right_only         0
Name: count, dtype: int64.

Weight captured in reconstructed trade database: 5687.9 of 6331.6 million tonnes in original 'trade_ex' dataset (89.8%).
Value captured in reconstructed trade database: $5519.2bn of $6537.5bn in original 'trade_ex' dataset (84.4%).
Energy Demand captured in reconstructed trade database: 1.4 of 8.8 EJs in original 'voys' dataset (15.9%).
CO2 emissions captured in reconstructed trade database: 107.4 of 669.9 million tonnes in original 'voys' dataset (16.0%).
2030 ETS Compliance Costs captured in reconstructed trade database: $5.7bn of $31.1bn in original 'voys' dataset (18.4%).
2030 NZF BAU Compliance Costs captured in reconstructed trade database: $57.7bn of $294.4bn in original 'voys' dataset (19.6%).
2030 NZF S24 Compliance Costs captured in reconstructed trade database: $65.4bn of

,trade_idx,source_iso_code,source_country,dest_iso_code,dest_country,HS2,mode_code,mode_label,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon,_recon_voys_info_merge
0,12,004,Afghanistan,032,Argentina,90,21,Sea,72.646,1.841,1.841,72.646,5.429581e-07,0.000042,0.0,0.000000,0.035696,0.035722,0.030116,0.028068,0.035696,0.039151,0.045789,0.047223,both
1,25,004,Afghanistan,040,Austria,12,21,Sea,114.923,183.004,183.004,114.923,5.246432e-05,0.004064,0.0,0.487683,2.205752,2.289774,1.854088,1.838388,2.205752,2.648882,3.000512,3.171319,both


## 2.6 Derive Aggregated Results Tables

In [15]:
trade_ex_voys_info_res_agg = {
    "val_usd":"sum", "vol_kg":"sum", "vol_kg_recon":"sum", "val_usd_recon":"sum", 
    "ene_tj_recon":"sum", "co2_t_recon":"sum", "ets_23_usd_recon":"sum", "ets_30_usd_recon":"sum",
    "bau_usd_23_recon":"sum", "bau_usd_30_recon":"sum", "bau_usd_40_recon":"sum", "bau_usd_50_recon":"sum",
    "s24_usd_23_recon":"sum", "s24_usd_30_recon":"sum", "s24_usd_40_recon":"sum", "s24_usd_50_recon":"sum"
}

# Full Disaggregation by Export and Partner Country and HS2 grouping
trade_ex_voys_info_res_g_full_c = ["source_iso_code", "source_country", "dest_iso_code", "dest_country", "HS2"]
trade_ex_voys_info_res_g_full = trade_ex_voys_info.groupby(trade_ex_voys_info_res_g_full_c).agg(trade_ex_voys_info_res_agg).reset_index()
trade_ex_voys_info_res_g_full.to_csv(exp_side_dir + "trade_ex_voys_info_res_g_full.csv", index=False)
display(trade_ex_voys_info_res_g_full.head(4))

# Disaggregation by Exporting Country and HS2
trade_ex_voys_info_res_g_coun_hs2_c = ["source_iso_code", "source_country", "HS2"]
trade_ex_voys_info_res_g_coun_hs2 = trade_ex_voys_info.groupby(trade_ex_voys_info_res_g_coun_hs2_c).agg(trade_ex_voys_info_res_agg).reset_index()
trade_ex_voys_info_res_g_coun_hs2.to_csv(exp_side_dir + "trade_ex_voys_info_res_g_coun_hs2.csv", index=False)
display(trade_ex_voys_info_res_g_coun_hs2.head(4))

# Disaggregation by Exporting Country only
trade_ex_voys_info_res_g_coun_c = ["source_iso_code", "source_country"]
trade_ex_voys_info_res_g_coun = trade_ex_voys_info.groupby(trade_ex_voys_info_res_g_coun_c).agg(trade_ex_voys_info_res_agg).reset_index()
trade_ex_voys_info_res_g_coun.to_csv(exp_side_dir + "trade_ex_voys_info_res_g_coun.csv", index=False)
display(trade_ex_voys_info_res_g_coun.head(4))

# Disaggregation by HS2 Export Grouping
trade_ex_voys_info_res_g_hs2_c = ["HS2"]
trade_ex_voys_info_res_g_hs2 = trade_ex_voys_info.groupby(trade_ex_voys_info_res_g_hs2_c).agg(trade_ex_voys_info_res_agg).reset_index()
trade_ex_voys_info_res_g_hs2.to_csv(exp_side_dir + "trade_ex_voys_info_res_g_hs2.csv", index=False)
display(trade_ex_voys_info_res_g_hs2.head(4))

,source_iso_code,source_country,dest_iso_code,dest_country,HS2,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,004,Afghanistan,031,Azerbaijan,57,266.993,12.461,12.461,266.993,6.593665e-06,0.000511,0.0,0.000000,0.196676,0.175529,0.175148,0.162718,0.196676,0.217526,0.326834,0.339784
1,004,Afghanistan,032,Argentina,90,72.646,1.841,1.841,72.646,5.429581e-07,0.000042,0.0,0.000000,0.035696,0.035722,0.030116,0.028068,0.035696,0.039151,0.045789,0.047223
2,004,Afghanistan,040,Austria,12,114.923,183.004,183.004,114.923,5.246432e-05,0.004064,0.0,0.487683,2.205752,2.289774,1.854088,1.838388,2.205752,2.648882,3.000512,3.171319
3,004,Afghanistan,040,Austria,17,699.225,1059.633,1059.633,699.225,3.037798e-04,0.023532,0.0,2.823792,12.771786,13.258292,10.735573,10.644667,12.771786,15.337606,17.373620,18.362629


,source_iso_code,source_country,HS2,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,004,Afghanistan,02,19970.561,3789.508,3789.508,19970.561,0.000597,0.045780,0.0,0.0,11.313996,10.345822,9.952322,9.471194,11.313996,12.436627,17.303836,18.202640
1,004,Afghanistan,04,2592.219,969.702,969.702,2592.219,0.000048,0.003657,0.0,0.0,1.344396,1.219692,1.155249,1.090276,1.344396,1.463483,2.045604,2.153784
2,004,Afghanistan,05,1293777.132,523668.950,523668.950,1293777.132,0.144697,11.205177,0.0,0.0,4931.103808,4404.250880,4383.858723,4074.458024,4931.103808,5448.735130,8163.943425,8492.032505
3,004,Afghanistan,06,536.211,502.812,502.812,536.211,0.000265,0.020526,0.0,0.0,7.899658,7.050416,7.034852,6.535749,7.899658,8.737049,13.126473,13.646739


,source_iso_code,source_country,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,004,Afghanistan,1.697071e+08,3.215669e+08,3.215669e+08,1.697071e+08,12.663680,977.624399,0.0,5.961390e+02,4.793903e+05,4.618930e+05,4.050516e+05,3.874610e+05,4.793903e+05,5.354202e+05,6.758048e+05,7.112176e+05
1,008,Albania,4.759043e+08,1.820407e+09,1.820407e+09,4.759043e+08,213.669384,16440.066449,0.0,3.519236e+06,1.102927e+07,1.071760e+07,9.747937e+06,9.388475e+06,1.102927e+07,1.185084e+07,1.431403e+07,1.499985e+07
2,012,Algeria,2.488790e+10,5.387403e+10,5.387403e+10,2.488790e+10,9488.327445,729079.177004,0.0,5.023186e+07,4.412917e+08,4.303113e+08,3.796244e+08,3.704484e+08,4.412917e+08,4.787159e+08,5.743182e+08,6.091983e+08
3,016,American Samoa,4.718756e+05,2.035520e+06,2.035520e+06,4.718756e+05,0.215047,16.655369,0.0,0.000000e+00,1.091806e+04,1.127588e+04,9.202561e+03,9.116579e+03,1.091806e+04,1.304420e+04,1.488268e+04,1.572323e+04


,HS2,val_usd,vol_kg,vol_kg_recon,val_usd_recon,ene_tj_recon,co2_t_recon,ets_23_usd_recon,ets_30_usd_recon,bau_usd_23_recon,bau_usd_30_recon,bau_usd_40_recon,bau_usd_50_recon,s24_usd_23_recon,s24_usd_30_recon,s24_usd_40_recon,s24_usd_50_recon
0,02,1.823558e+10,6.843840e+09,6.843840e+09,1.823558e+10,1661.599310,127742.743816,0.0,9.453475e+06,4.949799e+07,4.470120e+07,4.113755e+07,3.928007e+07,4.949799e+07,5.514646e+07,7.503444e+07,7.896723e+07
1,03,4.766161e+10,1.125116e+10,1.125116e+10,4.766161e+10,3356.710572,258858.272972,0.0,1.141802e+07,1.004482e+08,9.010102e+07,8.359609e+07,7.975737e+07,1.004482e+08,1.113796e+08,1.522634e+08,1.600184e+08
2,04,3.279712e+10,1.355512e+10,1.355512e+10,3.279712e+10,2701.678495,207638.655779,0.0,1.804150e+07,7.784614e+07,6.954426e+07,6.493219e+07,6.190339e+07,7.784614e+07,8.624983e+07,1.179014e+08,1.237023e+08
3,05,4.059533e+09,3.571166e+09,3.571166e+09,4.059533e+09,605.293271,46372.974982,0.0,4.821657e+06,1.799879e+07,1.620747e+07,1.520439e+07,1.453918e+07,1.799879e+07,1.999579e+07,2.724373e+07,2.860334e+07
